# Example 15 — Square-Dalitz efficiency and background histograms

This example uses ROOT TH2 maps defined directly in $(m',\theta')$ for $B^+\to K^+\pi^+\pi^-$.


In [ ]:
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import uproot
from dalitzplotfitter import (DecayChannel,DecayModel,NonResonant,RealImag,Resonance,Parameter,Minimizer,BackgroundCategory,MultiBackgroundNLL,enable_x64,invariants_to_square_dalitz,square_dalitz_efficiency_from_root,square_dalitz_background_from_root,weighted_resample)
enable_x64()
channel=DecayChannel('B+',('K+','pi+','pi-'))
model=DecayModel(channel,[Resonance('Kstar892',(0,2),RealImag(1,0),mass=0.8958,width=0.0474,spin=1),Resonance('rho770',(1,2),RealImag(0.65,0.10),mass=0.7753,width=0.1491,spin=1),NonResonant(RealImag(-0.5,0.1))],normalization_method='square-dalitz',normalization_resolution=300,normalization_pair=(0,2))


## 1. Create ROOT TH2 maps in Square Dalitz coordinates


In [ ]:
n=30; edges=np.linspace(0,1,n+1); c=0.5*(edges[:-1]+edges[1:]); mp,tp=np.meshgrid(c,c,indexing='ij')
eff_values=0.55+0.35*(1-0.7*mp)*(0.75+0.25*np.cos(np.pi*(tp-0.5)))
bkg_values=0.25+1.6*np.exp(-0.5*((mp-0.72)/0.13)**2)+0.8*(1-tp)**2
path=Path('example15_sdp_maps.root')
with uproot.recreate(path) as f:
    f['efficiency_sdp']=(eff_values,edges,edges)
    f['background_sdp']=(bkg_values,edges,edges)
print(path.resolve())


## 2. Load the TH2 maps as native Square-Dalitz models


In [ ]:
kwargs=dict(mother_mass=channel.parent_mass,masses=channel.daughter_masses,pair=(0,2))
eff=square_dalitz_efficiency_from_root(path,'efficiency_sdp',**kwargs)
bkg=square_dalitz_background_from_root(path,'background_sdp',**kwargs)
fig,axs=plt.subplots(1,2,figsize=(12,4.8))
im0=axs[0].imshow(eff_values.T,origin='lower',extent=(0,1,0,1),aspect='auto'); axs[0].set(xlabel="m'",ylabel=r"$\theta'$",title='Efficiency in Square Dalitz'); fig.colorbar(im0,ax=axs[0])
im1=axs[1].imshow(bkg_values.T,origin='lower',extent=(0,1,0,1),aspect='auto'); axs[1].set(xlabel="m'",ylabel=r"$\theta'$",title='Background in Square Dalitz'); fig.colorbar(im1,ax=axs[1]); plt.show()


## 3. Evaluate the same maps on ordinary Dalitz points

The fitter supplies $(s_{12},s_{13},s_{23})$; the histogram objects internally convert them to $(m',\theta')$.


In [ ]:
pool=model.generate_phase_space(150000,seed=15001); d=pool.as_dict(); e=np.asarray(eff(d)); b=np.asarray(bkg(d))
fig,axs=plt.subplots(1,2,figsize=(12,4.8))
h0=axs[0].hist2d(np.asarray(pool.s13),np.asarray(pool.s23),bins=70,weights=e); axs[0].set(xlabel=r'$s_{13}$ [GeV$^2$]',ylabel=r'$s_{23}$ [GeV$^2$]',title='SDP efficiency evaluated on Dalitz plane'); fig.colorbar(h0[3],ax=axs[0])
h1=axs[1].hist2d(np.asarray(pool.s13),np.asarray(pool.s23),bins=70,weights=b); axs[1].set(xlabel=r'$s_{13}$ [GeV$^2$]',ylabel=r'$s_{23}$ [GeV$^2$]',title='SDP background evaluated on Dalitz plane'); fig.colorbar(h1[3],ax=axs[1]); plt.show()


## 4. Generate signal + background using the SDP maps and fit the signal fraction


In [ ]:
fs_true=0.80; N=30000; ns=int(N*fs_true); nb=N-ns
sig=weighted_resample(jax.random.key(15002),pool,pool.weights*model.intensity(d)*eff(d),ns,replace=True)
bg=weighted_resample(jax.random.key(15003),pool,pool.weights*bkg(d),nb,replace=True)
from dalitzplotfitter.kinematics import PhaseSpaceSample
data=PhaseSpaceSample(s12=jnp.concatenate([sig.s12,bg.s12]),s13=jnp.concatenate([sig.s13,bg.s13]),s23=jnp.concatenate([sig.s23,bg.s23]),weights=jnp.ones(N))
pdf=model.pdf(efficiency=eff); dd=data.as_dict(); norm=model.normalization_sample
bkg_norm=jnp.mean(norm.weights*bkg(norm.as_dict()))
fs=Parameter('signal_fraction',0.68,bounds=(0.01,0.99),step=0.01)
cat=BackgroundCategory('combinatorial',bkg(dd),bkg_norm)
nll=MultiBackgroundNLL(signal_density=lambda v:pdf(dd,v),backgrounds=(cat,),signal_fraction=fs)
res=Minimizer(nll,(fs,),verbose=1).fit(start_values={'signal_fraction':0.68},simplex=True,ncall=10000)
print('generated signal fraction:',fs_true); print('start:',0.68); print('fitted:',float(res.values['signal_fraction'])); print('valid:',res.valid)


## 5. Plot the selected data in Square Dalitz coordinates


In [ ]:
mp_data,tp_data=invariants_to_square_dalitz(data.s12,data.s13,data.s23,mother_mass=channel.parent_mass,masses=channel.daughter_masses,pair=(0,2))
plt.figure(figsize=(6.5,5.5)); plt.hist2d(np.asarray(mp_data),np.asarray(tp_data),bins=60,range=[[0,1],[0,1]]); plt.xlabel("m'"); plt.ylabel(r"$\theta'$"); plt.title('Toy data in Square Dalitz coordinates'); plt.colorbar(label='events'); plt.show()
